# Choosing the active space

> **Chapter focus** &mdash; Which electrons and orbitals must the calculation treat explicitly?

## Learning objectives

After completing this chapter, you will be able to:

- Explain why strongly correlated systems require more than one determinant.
- Distinguish inactive, active, and virtual orbitals.
- Select a valence active space with QDK/Chemistry.
- Explain how natural orbitals and reduced density matrices support active-space selection.
- Explain how orbital entropies can refine an active-space choice.
- Evaluate the tradeoff between active-space accuracy and problem size.

> **Lab notebook assignment**
>
> Record the evidence used to select the active space, not only the final orbital and electron
> counts. Explain what the energy comparison establishes within the chosen basis and identify the
> energy that will serve as the algorithmic reference.

## Before you begin

This course requires a Python environment with the **`qdk-chemistry[jupyter]`** package.

`qdk-chemistry` ships compiled binaries and is not available for native Windows. On Windows, run
this course inside WSL. Run the cell below to check the current environment.

In [1]:
from _unit import check_env

check_env()

✅,Python version,3.12.13
✅,Course venv,/home/dhaipatel/qdk-chem/.venv
✅,Packages,"qdk_chemistry, qdk_chemistry.utils.cubegen, pyscf"


## Setting up

This chapter resumes the stretched N<sub>2</sub> workflow from *Describing the molecule*. The cell
below reproduces that starting point: it builds the molecule, then solves the cc-pVDZ Hartree&ndash;Fock
problem whose wavefunction every later step consumes.

In [2]:
from qdk_chemistry.algorithms import create
from qdk_chemistry.data import Structure
from qdk_chemistry.data.symmetry import SymmetryLabel, axes
from qdk_chemistry.utils import Logger, compute_valence_space_parameters

Logger.set_global_level(Logger.LogLevel.off)

structure = Structure.from_xyz("""\
2
Stretched N2 molecule for the ground-state QPE tutorial
N    0.000000    0.000000    0.000000
N    0.000000    0.000000    1.850000
""")
charge = 0
spin_multiplicity = 1
basis_set = "cc-pvdz"

scf_solver = create("scf_solver", "qdk")
hartree_fock_energy, hartree_fock_wavefunction = scf_solver.run(
    structure,
    charge=charge,
    spin_multiplicity=spin_multiplicity,
    basis_or_guess=basis_set,
)

print(f"Hartree-Fock energy: {hartree_fock_energy:.12f} Hartree")

Hartree-Fock energy: -108.418633697214 Hartree


## The limits of one determinant

The Hartree&ndash;Fock method restricts the wavefunction to one optimized Slater determinant. This
determinant represents one electron configuration, a pattern of occupied spin orbitals. This
description is often a useful starting point near an equilibrium geometry, where one configuration
dominates the ground-state wavefunction.

Stretching a chemical bond can make several configurations similar in energy because electrons can
no longer be assigned adequately to one fixed pattern of occupied and unoccupied molecular orbitals.
The need to combine these multiple important configurations is called **static correlation**. The
stretched N<sub>2</sub> geometry has been selected to demonstrate this regime. The correlated
calculations below evaluate static correlation by constructing a multi-determinant wavefunction and
measuring orbital-occupation entropies.

A configuration interaction (CI) calculation addresses this limitation by calculating a wavefunction
expanded in multiple Slater determinants:

$$\vert \Psi \rangle = \sum_i c_i \vert \Phi_i \rangle,$$

where $\vert \Phi_i \rangle$ is determinant $i$ and $c_i$ is its coefficient. Allowing every
possible determinant in all 28 `cc-pvdz` spatial orbitals would be unnecessarily expensive for this
tutorial. An active-space model restricts which orbital occupations are allowed to vary, reducing
the computational costs.

## The active space

An active-space calculation partitions the spatial molecular orbitals into three groups:

- **Inactive orbitals** remain doubly occupied in every determinant. Their electrons contribute to
  the energy, but their occupations do not vary.
- **Active orbitals** may be empty, singly occupied, or doubly occupied in different determinants.
  The calculation treats correlation among the active electrons explicitly.
- **Virtual orbitals** remain empty in every determinant and do not participate explicitly in the
  correlated calculation.

<div style="border-left:4px solid #5aa9e6;background:rgba(90,169,230,0.10);padding:0.2em 1em;margin:1em 0;border-radius:4px;">
<details>
<summary>&#10067;&nbsp; <b>Why can an inactive orbital still contribute to the molecular energy?</b></summary>

<br>

An inactive spatial orbital remains doubly occupied in every determinant. Its electrons and their
interactions contribute to the core part of the active-space Hamiltonian even though the calculation
does not vary their occupations.

</details>
</div>

A complete active space containing $n_e$ active electrons in $n_o$ active spatial orbitals is
written CAS$(n_e,n_o)$. Complete active space configuration interaction (CASCI) forms every
determinant consistent with those active electron and orbital counts while keeping the molecular
orbitals fixed. Unlike complete active space self-consistent field (CASSCF), CASCI does not
reoptimize the orbitals.

A useful first choice is a generous valence space containing orbitals on both sides of the
occupied&ndash;virtual boundary. The `compute_valence_space_parameters` function determines the numbers of
valence electrons and valence spatial orbitals from the Hartree&ndash;Fock wavefunction and molecular
charge. The `qdk_valence` selector uses those numbers to construct an initial active space from
orbitals near the HOMO&ndash;LUMO gap:

In [3]:
num_valence_electrons, num_valence_orbitals = compute_valence_space_parameters(
    hartree_fock_wavefunction, charge
)
valence_selector = create(
    "active_space_selector",
    "qdk_valence",
    num_active_electrons=num_valence_electrons,
    num_active_orbitals=num_valence_orbitals,
)
valence_wavefunction = valence_selector.run(hartree_fock_wavefunction)

# Restricted alpha and beta channels contain the same spatial-orbital indices,
# so read one channel to count each spatial orbital once.
alpha_channel = SymmetryLabel([axes.alpha()])
valence_indices = list(
    valence_wavefunction.get_orbitals().active_indices().indices(alpha_channel)
)
num_valence_alpha, num_valence_beta = valence_wavefunction.get_active_num_electrons()

print(f"Initial valence space: CAS({num_valence_electrons}e, {num_valence_orbitals}o)")
print(f"Active alpha/beta electrons: {num_valence_alpha}/{num_valence_beta}")
print(f"Initial active orbital indices: {valence_indices}")

Initial valence space: CAS(10e, 8o)
Active alpha/beta electrons: 5/5
Initial active orbital indices: [2, 3, 4, 5, 6, 7, 8, 9]


For this restricted calculation, matching $\alpha$ and $\beta$ channels describe the same spatial
orbitals, so the code reads one channel and counts each spatial orbital once. Use these values with
the total number of `cc-pvdz` molecular orbitals from *Describing the molecule* to determine the
initial partitioning of inactive, active, and virtual orbitals.

## A correlated active-space wavefunction

The active-space selector labels orbitals but does not determine how strongly each orbital
participates in correlation. That evidence must come from a correlated wavefunction. The following
cell constructs the molecular Hamiltonian in the initial valence space and solves it with the MACIS
(Many-body Adaptive Configuration Interaction Solver) CASCI implementation:

In [4]:
hamiltonian_constructor = create("hamiltonian_constructor")
casci_solver = create(
    "multi_configuration_calculator",
    "macis_cas",
    # autoCAS entropies require both one- and two-particle RDMs.
    calculate_one_rdm=True,
    calculate_two_rdm=True,
)

valence_hamiltonian = hamiltonian_constructor.run(valence_wavefunction.get_orbitals())
valence_energy, valence_casci_wavefunction = casci_solver.run(
    valence_hamiltonian,
    num_valence_alpha,
    num_valence_beta,
)
num_valence_determinants = len(valence_casci_wavefunction.get_coefficients())

print(f"Initial CASCI energy: {valence_energy:.12f} Hartree")
print(f"Initial CASCI determinants: {num_valence_determinants}")

Initial CASCI energy: -108.778369520882 Hartree
Initial CASCI determinants: 3136


For an active space with $n_o$ spatial orbitals, $n_\alpha$ active $\alpha$ electrons, and
$n_\beta$ active $\beta$ electrons, choosing the occupied $\alpha$ and $\beta$ spin orbitals gives
independent counts that multiply. The number of possible determinants in the wavefunction is
therefore

$$N_{\mathrm{det}} = \binom{n_o}{n_\alpha}\binom{n_o}{n_\beta}.$$

The initial valence space in this example is small enough to include every determinant rather than
approximating the wavefunction with a selected subset. Larger active-space studies often use
selected CI (SCI) to obtain approximate active-space diagnostics at lower cost, but that additional
approximation is unnecessary here.

The `calculate_one_rdm` and `calculate_two_rdm` settings request the one- and two-particle reduced
density matrices (RDMs). An RDM compresses information from the many-electron wavefunction into
expectation values involving one or two particles. The spin-resolved one-particle RDM tracks
$\alpha$ and $\beta$ occupations separately, and its diagonal gives the expected occupation of each
spin orbital. A corresponding diagonal element of the two-particle RDM gives the joint occupation of
a pair of spin orbitals.

Each determinant assigns every spatial orbital one of four *local occupation states*: empty,
occupied by one $\alpha$ electron, occupied by one $\beta$ electron, or doubly occupied. Here, a
local occupation state describes only one orbital, not the complete electronic state of the
molecule. For these occupation quantities, each determinant contributes according to the squared
magnitude of its coefficient in the correlated wavefunction. Together, the RDM elements collect
these contributions into the probabilities of the four local occupation states. If the important
determinants give an orbital the same occupation, one probability dominates; if they assign
different occupations, the probabilities spread among several local states.

## Natural-orbital transformation

Molecular orbitals are not unique: a unitary rotation applied within an active orbital subspace
changes the individual orbital shapes but not the subspace they span. For an exact CASCI calculation
in a fixed active subspace, such a rotation leaves the total energy unchanged. Orbital-resolved
quantities, however, can change because they describe the chosen orbital representation.

Natural orbitals diagonalize the one-particle RDM. Their eigenvalues are natural-orbital occupation
numbers between zero and two for spatial orbitals. In this basis, the off-diagonal elements vanish,
so each occupation number is associated directly with one natural orbital rather than being mixed
among several orbitals. Occupations near two identify nearly doubly occupied orbitals, occupations
near zero identify nearly empty orbitals, and fractional occupations can reveal orbitals that
require multiple electron configurations.

The supported `qdk_natural_orbitals` transformation uses the one-particle RDM from the initial CASCI
wavefunction. It rotates the active orbitals into the natural-orbital representation described
above. The next cell then rebuilds and resolves the initial valence-space Hamiltonian so that both
RDMs and the orbital diagnostics are expressed consistently in the natural-orbital representation:

In [5]:
# Rotate the valence orbitals using the CASCI one-particle RDM so each
# natural orbital has a well-defined correlated occupation.
natural_orbital_localizer = create("orbital_localizer", "qdk_natural_orbitals")
natural_orbital_wavefunction = natural_orbital_localizer.run(
    valence_casci_wavefunction,
    valence_indices,
    valence_indices,
)

# Rebuild and solve in the rotated basis so the RDMs and orbital entropies
# describe the same natural-orbital representation.
natural_orbital_hamiltonian = hamiltonian_constructor.run(natural_orbital_wavefunction.get_orbitals())
natural_orbital_energy, natural_orbital_casci_wavefunction = casci_solver.run(
    natural_orbital_hamiltonian,
    num_valence_alpha,
    num_valence_beta,
)
# Store ordinary Python floats rather than library scalar types so the
# values can be printed and passed to the visualization cells directly.
orbital_entropies = [
    float(value) for value in natural_orbital_casci_wavefunction.get_single_orbital_entropies()
]

print(f"Natural-orbital CASCI energy: {natural_orbital_energy:.12f} Hartree")
print(f"Energy change: {natural_orbital_energy - valence_energy:.12e} Hartree")

Natural-orbital CASCI energy: -108.778369520882 Hartree
Energy change: 1.421085471520e-14 Hartree


The two CASCI energies should agree to the displayed precision because both calculations span the
same complete active space. The second calculation is needed for the orbital-resolved selection
evidence, not to lower the energy.

## Active-space refinement with orbital entropies

The single-orbital entropy used here is the von Neumann entropy of the reduced density matrix for
one spatial orbital. Its eigenvalues are the probabilities $\omega_{a,i}$ of the four local
occupation states, so the entropy has the Shannon form

$$s_i^{(1)} = -\sum_{a=1}^{4} \omega_{a,i}\ln\omega_{a,i}.$$

These probabilities come from diagonal elements of the spin-resolved one- and two-particle RDMs. If
$n_{i\alpha}$ and $n_{i\beta}$ are the one-particle occupations and $d_i$ is the *double-occupancy
probability* &mdash; the probability that the $\alpha$ and $\beta$ spin orbitals belonging to spatial
orbital $i$ are occupied simultaneously &mdash; then

$$\begin{aligned}
\omega_{\mathrm{empty},i} &= 1-n_{i\alpha}-n_{i\beta}+d_i, \\
\omega_{\alpha,i} &= n_{i\alpha}-d_i, \\
\omega_{\beta,i} &= n_{i\beta}-d_i, \\
\omega_{\mathrm{double},i} &= d_i.
\end{aligned}$$

An entropy near zero means that one local occupation dominates. A larger entropy means that several
local occupations occur across the important determinants. These high-entropy orbitals carry the
strongest static-correlation signal because their occupations vary among the important determinants.
Freezing a high-entropy orbital would prevent its occupation from changing with the occupations of
the other orbitals and would therefore remove an important part of the multi-configurational
wavefunction. By contrast, a low-entropy orbital remains close to one local occupation state and is
a better candidate to freeze as inactive or virtual.

<div style="border-left:4px solid #5aa9e6;background:rgba(90,169,230,0.10);padding:0.2em 1em;margin:1em 0;border-radius:4px;">
<details>
<summary>&#10067;&nbsp; <b>Why does autoCAS require a correlated calculation before it can select orbitals?</b></summary>

<br>

The selector uses single-orbital entropies derived from local occupation probabilities. Those
probabilities require one- and two-particle RDMs from a correlated wavefunction; a Hartree&ndash;Fock
determinant alone does not provide the required correlation evidence.

</details>
</div>

The entropy-difference autoCAS selector, `qdk_autocas_eos`, sorts the normalized orbital entropies
and tests the consecutive gaps against its entropy and difference thresholds. It selects the largest
high-entropy group separated by a qualifying gap. A large gap provides evidence of a natural
boundary between orbitals with similarly strong occupation coupling and orbitals whose occupations
are much less coupled to the rest of the active space. It then repartitions the orbitals according
to the selected group:

In [6]:
# autoCAS uses the RDM-derived orbital entropies to retain the orbitals that
# carry the strongest correlation in a smaller active space.
autocas_selector = create("active_space_selector", "qdk_autocas_eos")
refined_wavefunction = autocas_selector.run(natural_orbital_casci_wavefunction)
refined_orbitals = refined_wavefunction.get_orbitals()

# Summarize the inactive, selected active, and virtual spatial-orbital spaces.
alpha_channel = SymmetryLabel([axes.alpha()])
refined_indices = list(refined_orbitals.active_indices().indices(alpha_channel))
inactive_indices = list(refined_orbitals.inactive_indices().indices(alpha_channel))
num_refined_alpha, num_refined_beta = refined_wavefunction.get_active_num_electrons()
num_refined_electrons = num_refined_alpha + num_refined_beta
num_refined_orbitals = len(refined_indices)
num_virtual_orbitals = (
    refined_orbitals.get_num_molecular_orbitals() - len(inactive_indices) - num_refined_orbitals
)

print("Single-orbital entropies (* = selected by autoCAS):")
for index, entropy in zip(valence_indices, orbital_entropies):
    marker = " * " if index in refined_indices else "   "
    print(f"{marker}orbital {index}: {entropy:.9f}")
print(f"Refined active space: CAS({num_refined_electrons}e, {num_refined_orbitals}o)")
print(f"Inactive orbital indices: {inactive_indices}")
print(f"Virtual orbitals: {num_virtual_orbitals}")

Single-orbital entropies (* = selected by autoCAS):
   orbital 2: 0.021695611
   orbital 3: 0.029962798
 * orbital 4: 0.547855595
 * orbital 5: 0.963884196
 * orbital 6: 0.963884196
 * orbital 7: 0.966011187
 * orbital 8: 0.966011187
 * orbital 9: 0.554009272
Refined active space: CAS(6e, 6o)
Inactive orbital indices: [0, 1, 2, 3]
Virtual orbitals: 18


The asterisks in the output identify the selected orbitals. The selected high-entropy group
determines the refined active space. Among the unselected orbitals, those below the occupied&ndash;virtual
boundary of the reference determinant become inactive, while those above the boundary become
virtual. Freezing these low-entropy orbitals is still an approximation because low entropy does not
mean that their correlation contribution is exactly zero, so the energy comparison below measures
part of its cost.

## The algorithmic reference

The workflow finishes by solving the refined active-space Hamiltonian with CASCI:

In [7]:
refined_hamiltonian = hamiltonian_constructor.run(refined_orbitals)
refined_energy, refined_casci_wavefunction = casci_solver.run(
    refined_hamiltonian,
    num_refined_alpha,
    num_refined_beta,
)
num_refined_determinants = len(refined_casci_wavefunction.get_coefficients())

print(f"Final CASCI energy: {refined_energy:.12f} Hartree")
print(f"Final CASCI determinants: {num_refined_determinants}")
print(
    "Energy increase from reducing the active space: "
    f"{refined_energy - natural_orbital_energy:.12f} Hartree"
)

Final CASCI energy: -108.771051792966 Hartree
Final CASCI determinants: 400
Energy increase from reducing the active space: 0.007317727916 Hartree


The resulting determinant count quantifies the reduction in problem size for the quantum-computing
stages of the tutorial. The final CASCI energy is the exact ground-state energy of the selected
active-space Hamiltonian, up to numerical solver tolerance, and will be the *algorithmic reference
energy* for state preparation and phase estimation. CASCI is a full configuration-interaction
calculation within the selected active space, but it is not the exact energy of N<sub>2</sub> in the
full `cc-pvdz` orbital space: fixing the inactive orbitals as doubly occupied and the virtual
orbitals as empty excludes correlation involving those orbitals.

## The active-space choice

The initial valence space includes more orbitals than the refined active space so that the
correlated calculation can first measure the entropy of every candidate orbital. The refinement then
uses this evidence to decide which orbital occupations must remain variable and which can be frozen.

Freezing additional orbital occupations cannot lower the CASCI energy. It leaves the energy
unchanged only if the removed determinants contribute nothing to the larger-space ground state;
otherwise, as in this example, the energy increases.

The observed increase quantifies correlation excluded when reducing the initial valence space. This
active-space model error is separate from the 1 milliHartree teaching target, which applies only to
the later quantum algorithm's agreement with the compact-model CASCI reference.

<div style="border-left:4px solid #5aa9e6;background:rgba(90,169,230,0.10);padding:0.2em 1em;margin:1em 0;border-radius:4px;">
<details>
<summary>&#10067;&nbsp; <b>Why should the energy increase caused by active-space refinement not be judged against the 1 milliHartree teaching target?</b></summary>

<br>

The energy increase measures correlation excluded when orbital occupations are frozen during
active-space refinement. The 1 milliHartree target applies later when comparing the
phase-estimation energy with the exact CASCI energy of the same selected-space Hamiltonian. These
comparisons measure different approximations.

</details>
</div>

For this tutorial, the refined active space is accepted as a compact model because it retains the
orbitals with the strongest entropy-based correlation evidence while producing a tractable
Hamiltonian for validating the quantum workflow. The energy difference from the initial
valence-space calculation remains documented as model error. The next chapter will determine how the
selected active spatial orbitals are represented by qubits.

## Count the determinants

The refined active space is CAS$(6,6)$: six electrons in six spatial orbitals, so three $\alpha$ and
three $\beta$ electrons. The $\alpha$ and $\beta$ occupations are chosen independently of each
other.

Fix the function below so that it returns the number of determinants this active space contains,
then run the cell.

In [ ]:
from math import comb

from _unit import exercise


@exercise
def determinant_count():
    alpha = comb(6, 3)
    beta = comb(6, 3)
    return alpha

Every way of placing the three $\alpha$ electrons can be paired with every way of placing the three
$\beta$ electrons, so the two counts combine multiplicatively rather than additively.

Return the product of the two counts:

```python
return alpha * beta
```

$\binom{6}{3}=20$, so the refined space holds $20\times20=400$ determinants, down from the 3,136 of
the initial CAS$(10,8)$ valence space.

## Check your understanding

<div style="border-left:4px solid #5aa9e6;background:rgba(90,169,230,0.10);padding:0.2em 1em;margin:1em 0;border-radius:4px;">
<details>
<summary>&#10067;&nbsp; <b>What initial valence space and determinant count did the workflow construct?</b></summary>

<br>

Ten active electrons in eight active spatial orbitals, written CAS$(10,8)$, with five $\alpha$ and
five $\beta$ active electrons. The active orbital indices are 2 through 9. Of the 28 `cc-pvdz`
molecular orbitals, indices 0 and 1 are initially inactive and the remaining 18 are virtual. The
determinant count is $\binom{8}{5}\binom{8}{5}=3136$.

</details>
</div>

<div style="border-left:4px solid #5aa9e6;background:rgba(90,169,230,0.10);padding:0.2em 1em;margin:1em 0;border-radius:4px;">
<details>
<summary>&#10067;&nbsp; <b>Which orbitals did autoCAS retain, and how much did refinement reduce the problem size?</b></summary>

<br>

Orbitals 4 through 9 have the largest entropies, separated by a large gap from the remaining
values. autoCAS retains these six orbitals in CAS$(6,6)$, containing three $\alpha$ and three
$\beta$ active electrons. The refined partition has four inactive orbitals, six active orbitals, and
18 virtual orbitals. Its determinant count is $\binom{6}{3}\binom{6}{3}=400$, compared with 3,136
determinants in the initial valence space.

</details>
</div>

<div style="border-left:4px solid #5aa9e6;background:rgba(90,169,230,0.10);padding:0.2em 1em;margin:1em 0;border-radius:4px;">
<details>
<summary>&#10067;&nbsp; <b>Did the natural-orbital transformation change the CASCI energy?</b></summary>

<br>

No change appears at the displayed precision. The reported signed energy change after the
transformation is consistent with numerical roundoff near zero. Both calculations span the same
complete active subspace, so changing the orbital representation does not change the exact CASCI
energy within that subspace.

</details>
</div>

Record the orbital representation, initial and refined active-space sizes, selection evidence,
determinant counts, and both CASCI energies in the active-space section of the lab notebook. Use the
final selected-space energy as the algorithmic reference, while retaining the larger-space result as
evidence of the correlation excluded by the compact model.

## Candidate-orbital visualization

The molecular viewer needs values of each orbital's spatial wavefunction on a three-dimensional
grid. The next cell evaluates all candidate natural orbitals on such a grid and stores the sampled
values as cube data.

Each orbital is annotated with its natural occupation, single-orbital entropy, and autoCAS selection
status so that you can examine the spatial and numerical evidence together. Generating the orbital
grids may take a little longer than the preceding calculation.

In [9]:
from qdk_chemistry.utils.cubegen import generate_cubefiles_from_orbitals

natural_orbitals = natural_orbital_casci_wavefunction.get_orbitals()
occupation_alpha, occupation_beta = (
    natural_orbital_casci_wavefunction.get_active_orbital_occupations()
)

# Occupation arrays use active-space positions, while cube-file labels use
# the original molecular-orbital indices; this dictionary connects them.
active_position = {
    orbital_index: position for position, orbital_index in enumerate(valence_indices)
}
raw_cube_data = generate_cubefiles_from_orbitals(
    orbitals=natural_orbitals,
    grid_size=(30, 30, 30),
    margin=10.0,
    indices=valence_indices,
)

cube_data = {}
for raw_label, cube_file in raw_cube_data.items():
    # Cube labels number orbitals from one, while QDK/Chemistry indices start
    # from zero, so convert before looking up occupations and entropies.
    orbital_index = int(raw_label.split("_")[1]) - 1
    position = active_position[orbital_index]
    # Add the alpha and beta occupations to report the total occupation of
    # each spatial orbital in the viewer.
    occupation = float(occupation_alpha[position]) + float(occupation_beta[position])
    cube_data[f"Orbital {orbital_index}"] = {
        "data": cube_file,
        "info": {
            "Occupation": f"{occupation:.3f}",
            "Entropy": f"{orbital_entropies[position]:.3f}",
            "Selected by autoCAS": "yes" if orbital_index in refined_indices else "no",
        },
    }

print(f"Generated cube data for {len(cube_data)} candidate orbitals.")

Generated cube data for 8 candidate orbitals.


## Inspect the natural orbitals

Launch the interactive molecular viewer and use its orbital menu to move through the candidate
natural orbitals. For each orbital, inspect the isosurface together with the displayed "occupation",
"entropy", and "selected by autoCAS" information. Compare the selected orbitals with the nearly
doubly occupied and nearly empty orbitals that were excluded.

In [10]:
from qdk.widgets import MoleculeViewer

MoleculeViewer(
    molecule_data=structure.to_xyz(),
    cube_data=cube_data,
)

Use the viewer to inspect the following information:

- **Orbital menu** selects each candidate natural orbital for comparison. The menu follows
  increasing molecular-orbital index, which corresponds here to decreasing natural occupation. The
  menu is not ordered by entropy.
- **Isosurface** traces points where the orbital wavefunction has a chosen positive or negative
  value, revealing its lobes, nodes, and spatial extent. The surface itself does not encode
  occupation or entropy.
- **Natural occupation** reports the average number of electrons in the spatial orbital. A value
  near two indicates an almost always doubly occupied orbital, a value near zero indicates an almost
  always empty orbital, and an intermediate value indicates variable occupation across the
  correlated wavefunction.
- **Single-orbital entropy and autoCAS selection** report the uncertainty in the orbital's local
  occupation and whether autoCAS retained it. Larger entropy indicates stronger coupling to the
  occupations of the other active orbitals.

autoCAS selects the strongly coupled group from gaps in the orbital entropies, not from orbital
shapes or a cutoff applied to the natural occupations. Use the shapes as aids to chemical
interpretation, but defend the final active space using the numerical occupation and entropy
evidence in the overlays.

## Interpret the active-space choice

Identify which visual features distinguish nearly doubly occupied orbitals, high-entropy orbitals
with strongly coupled occupations, and nearly empty orbitals. Then explain why autoCAS retains the
selected group while excluding the other candidate orbitals.

Treat the orbital shapes as aids to chemical interpretation rather than as the selection rule.
Defend the refined active-space choice using the occupation and entropy overlays, and record your
explanation in the active-space section of the lab notebook.